In [2]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3
from botocore.exceptions import NoCredentialsError
from dotenv import load_dotenv

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Obter as credenciais do MinIO do arquivo .env
minio_url = os.getenv('MINIO_ENDPOINT')
minio_access_key = os.getenv('MINIO_ACCESS_KEY')
minio_secret_key = os.getenv('MINIO_SECRET_KEY')
bucket_name = "bronze"

# Criar cliente MinIO
s3_client = boto3.client('s3',
                         endpoint_url=f'http://{minio_url}',
                         aws_access_key_id=minio_access_key,
                         aws_secret_access_key=minio_secret_key,
                         config=boto3.session.Config(signature_version='s3v4'))

# Carregar dados limpos do notebook eda.ipynb
df_merged = pd.read_csv("../data/processed/dados_mesclados_limpos.csv")

# Verificar os primeiros registros dos dados limpos
print("Dados limpos carregados:")
print(df_merged.head())

# Criar a pasta bronze se não existir
os.makedirs("../data/bronze", exist_ok=True)

# Limpar os dados existentes na pasta bronze
bronze_files = os.listdir("../data/bronze")
for file in bronze_files:
    os.remove(os.path.join("../data/bronze", file))
print("Dados antigos removidos da pasta bronze.")

# Converter o DataFrame limpo para o formato Parquet para armazenamento eficiente
tabela = pa.Table.from_pandas(df_merged)
parquet_path = "../data/bronze/nafld_merged.parquet"
pq.write_table(tabela, parquet_path)

# Verificar se o arquivo Parquet foi salvo
print("Arquivos na pasta bronze:")
print(os.listdir("../data/bronze"))

# Carregar arquivo Parquet para MinIO
try:
    s3_client.upload_file(parquet_path, bucket_name, "nafld_merged.parquet")
    print("Arquivo Parquet enviado para o bucket MinIO com sucesso.")
except FileNotFoundError:
    print("Arquivo não encontrado.")
except NoCredentialsError:
    print("Credenciais do MinIO não encontradas.")

Dados limpos carregados:
   Unnamed: 0_x  id  age   male  weight  height        bmi  case.id  futime  \
0          3631   1   57  False    60.0   163.0  22.690939  10630.0    6261   
1          3631   1   57  False    60.0   163.0  22.690939  10630.0    6261   
2          3631   1   57  False    60.0   163.0  22.690939  10630.0    6261   
3          3631   1   57  False    60.0   163.0  22.690939  10630.0    6261   
4          3631   1   57  False    60.0   163.0  22.690939  10630.0    6261   

   status  Unnamed: 0_y  days  test  value  
0   False        135077  -459   hdl   75.0  
1   False        313143  -459  chol   75.0  
2   False        135078   183   hdl   64.0  
3   False        313144   183  chol   64.0  
4   False        135079  2030   hdl   74.0  
Dados antigos removidos da pasta bronze.
Arquivos na pasta bronze:
['nafld_merged.parquet']
Arquivo Parquet enviado para o bucket MinIO com sucesso.
